# Keras によるテキスト分類

**学習目標**
1. BigQuery を使用してテキスト分類データセットを作成する方法を学ぶ
1. Keras 前処理レイヤを使用して、トレーニング用のテキストコーパスをトークン化および整数化する方法を学ぶ
1. Keras で one-hot エンコーディングを実行する方法を学ぶ
1. Keras で単語を表現するために埋め込みレイヤを使用する方法を学ぶ
1. 文の Bag-of-Words 表現について学ぶ
1. Keras でテキストを分類するために DNN / CNN / RNN モデルを使用する方法を学ぶ

## はじめに


このノートブックでは、ラボの最初のタスクで構築したタイトル データセット内のタイトルから、その推定ソース（GitHub、TechCrunch、または The New York Times）を認識するテキストモデルを実装します。

次のステップでは、Keras モデルに入力できるようにテキストとラベルを読み込み、前処理を行います。タイトルのテキストについては、Keras 前処理レイヤ（Keras Preprocessing Layers）を使用してタイトルを前処理する方法を学びます。Keras 前処理レイヤを使用することで、未加工の生データをモデルに直接入力できるようになります。ラベルについては、3 つのクラスのそれぞれを 3 次元の基底ベクトルに one-hot エンコードする方法を学びます。

その後、タイトル分類を行ういくつかの可能なモデルについて見ていきます。

最初のモデルは、単語ベクトルを平均化し、得られたテンソルをさらに Dense レイヤに入力するシンプルな Bag-of-Words DNN モデルです。これにより単語の順序は失われます（したがって、文を「Bag-of-Words（単語の袋）」とみなします）。2 番目と 3 番目のモデルでは、シンプルな RNN とシンプルな CNN を使用して単語の順序に関する情報を保持し、DNN モデルと同等のパフォーマンスをはるかに少ないエポック数で達成できるようにします。

In [ ]:
import os

import pandas as pd
from google.cloud import bigquery

In [ ]:
%load_ext google.cloud.bigquery

下のセルの変数の値を置き換えてください:

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
%env PROJECT = {PROJECT}
%env BUCKET = {PROJECT}
%env REGION = "us-central1"
SEED = 0

## BigQuery からデータセットを作成する 

Hacker News の見出しは、BigQuery の一般公開データセットとして利用できます。[データセット](https://bigquery.cloud.google.com/table/bigquery-public-data:hacker_news.stories?tab=details) には、2006 年 10 月のサイト開設から 2015 年 10 月までのすべての見出しが含まれています。 

以下はデータセットのサンプルです:

In [ ]:
%%bigquery --project $PROJECT

SELECT
    url, title, score
FROM
    `bigquery-public-data.hacker_news.full`
WHERE
    LENGTH(title) > 10
    AND score > 10
    AND LENGTH(url) > 0
LIMIT 10

BigQuery で正規表現のパースを実行し、URL からニュース記事のソースを取得してみましょう。たとえば、URL が http://mobile.nytimes.com/.... の場合、<i>nytimes</i> だけを残したいと考えます。

In [ ]:
%%bigquery --project $PROJECT

SELECT
    ARRAY_REVERSE(SPLIT(REGEXP_EXTRACT(url, '.*://(.[^/]+)/'), '.'))[SAFE_OFFSET(1)] AS source,
    COUNT(title) AS num_articles
FROM
    `bigquery-public-data.hacker_news.full`
WHERE
    REGEXP_CONTAINS(REGEXP_EXTRACT(url, '.*://(.[^/]+)/'), '.com$')
    AND LENGTH(title) > 10
GROUP BY
    source
ORDER BY num_articles DESC
  LIMIT 100

URL を適切にパースしてソースを取得できるようになったので、ソースとタイトルのデータセットをまとめましょう。これが機械学習用のラベル付きデータセットになります。

In [ ]:
regex = ".*://(.[^/]+)/"


sub_query = """
SELECT
    title,
    ARRAY_REVERSE(SPLIT(REGEXP_EXTRACT(url, '{0}'), '.'))[SAFE_OFFSET(1)] AS source
    
FROM
    `bigquery-public-data.hacker_news.full`
WHERE
    REGEXP_CONTAINS(REGEXP_EXTRACT(url, '{0}'), '.com$')
    AND LENGTH(title) > 10
""".format(
    regex
)


query = """
SELECT 
    LOWER(REGEXP_REPLACE(title, '[^a-zA-Z0-9 $.-]', ' ')) AS title,
    source
FROM
  ({sub_query})
WHERE (source = 'github' OR source = 'nytimes' OR source = 'techcrunch')
""".format(
    sub_query=sub_query
)

print(query)

機械学習のトレーニングでは、通常、データセットをトレーニング用と評価用のデータセットに分割する必要があります（評価データセットに基づいてモデルや特徴量の選択を行う場合は、さらに独立したテスト データセットも必要になる場合があります）。

In [ ]:
bq = bigquery.Client(project=PROJECT)
title_dataset = bq.query(query).to_dataframe()
title_dataset.head()

In [ ]:
print(f"The full dataset contains {len(title_dataset)} titles")

3 つのラベルのそれぞれについて、ラベルの数がほぼ同じであることを確認しましょう:

In [ ]:
title_dataset.source.value_counts()

最後に、現在メモリ上にあるデータをディスクに保存します。

データセット全体を含む CSV ファイルを作成し、`data` フォルダに保存します。

In [ ]:
DATADIR = "./data/"

if not os.path.exists(DATADIR):
    os.makedirs(DATADIR)

In [ ]:
FULL_DATASET_NAME = "titles_full.csv"
FULL_DATASET_PATH = os.path.join(DATADIR, FULL_DATASET_NAME)

# ディスクに書き込む前にデータをシャッフルします。
title_dataset = title_dataset.sample(n=len(title_dataset))

title_dataset.to_csv(
    FULL_DATASET_PATH, header=False, index=False, encoding="utf-8"
)

## テキスト分類

データが CSV ファイルとして保存されたので、関連するライブラリをインポートすることから始めましょう。

In [ ]:
import os
import shutil

import keras
import pandas as pd
import tensorflow as tf
from keras import Input
from keras.callbacks import EarlyStopping, TensorBoard
from keras.layers import (
    GRU,
    Conv1D,
    Dense,
    Embedding,
    Flatten,
    Lambda,
    TextVectorization,
)
from keras.models import Sequential
from keras.utils import to_categorical

In [ ]:
%matplotlib inline

まず、トレーニング済みモデルに関する情報が保存される場所と、データセットが配置されている場所を指定します:

In [ ]:
LOGDIR = "./text_models"
DATA_DIR = "./data"

## データセットの読み込み

このデータセットは、記事のタイトルと、それらの記事がどのソース（GitHub、TechCrunch、または The New York Times）から取得されたかを示すラベルで構成されています。

In [ ]:
DATASET_NAME = "titles_full.csv"
TITLE_SAMPLE_PATH = os.path.join(DATA_DIR, DATASET_NAME)
COLUMNS = ["title", "source"]

titles_df = pd.read_csv(TITLE_SAMPLE_PATH, header=None, names=COLUMNS)
titles_df.head()

In [ ]:
titles_df.source[:4]

次に、以下の処理を行う関数を作成する必要があります。
* タイトルのソースを受け取る
* 対応する one-hot エンコードされたベクトルを返す

これには Keras の `to_categorical` が便利です。

In [ ]:
CLASSES = {"github": 0, "nytimes": 1, "techcrunch": 2}
N_CLASSES = len(CLASSES)

### ラボ課題

下の `encode_labels` 関数のコードを完成させて、以下の処理を行ってください。
* 上記の `CLASSES` 辞書を使用して、`sources` 内の各 `source` を対応する数値にマッピングするリストを作成する
* Keras 関数を使用して、変数のクラスを one-hot エンコードする

In [ ]:
# TODO


def encode_labels(sources):
    classes = None  # TODO
    one_hots = None  # TODO
    return one_hots

In [ ]:
encode_labels(titles_df.source[:4])

## トレーニング / テスト分割の準備

データをトレーニング用とテスト用の分割に分けましょう:

In [ ]:
DATASET_SIZE = titles_df.shape[0]
N_TRAIN = int(DATASET_SIZE * 0.80)

titles_train, sources_train = (
    titles_df.title[:N_TRAIN],
    titles_df.source[:N_TRAIN],
)

titles_valid, sources_valid = (
    titles_df.title[N_TRAIN:],
    titles_df.source[N_TRAIN:],
)

念のため、トレーニング用とテスト用の分割で、各クラスのサンプル数がほぼ同数であることを確認します。

同数であれば、モデルのパフォーマンスを測定するための適切な指標として正解率（accuracy）を使用できます。

In [ ]:
sources_train.value_counts()

In [ ]:
sources_valid.value_counts()

`encode_labels` 関数を使用して、モデルに入力するトレーニング データと検証データを準備できるようになりました。ラベルは one-hot エンコードされた 3 次元ベクトルになります。

In [ ]:
Y_train = keras.ops.convert_to_tensor(encode_labels(sources_train))
Y_valid = keras.ops.convert_to_tensor(encode_labels(sources_valid))

In [ ]:
Y_train[:3]

## Keras テキスト前処理レイヤ

モデルにデータを渡し始める前に、テキストを前処理する必要があります。Keras 前処理レイヤを使用することで、これをモデル内に直接組み込むことができます。

実行する前処理:
- テキストのトークン化
- トークンの整数化

インスタンス化の際に、出力シーケンスの最大長と、考慮するトークンの最大数を指定できます。

### ラボ課題

Keras 前処理レイヤである [TextVectorization](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/) レイヤを使用して、テキスト特徴量を整数シーケンスにマッピングしてテキストをベクトル化します。

In [ ]:
MAX_LEN = 26
MAX_TOKENS = 20000

preprocessing_layer = #TODO: your code goes here

このレイヤをモデルで使用する前に、トークンから整数へのマッピングを生成するようにデータに適応（adapt）させる必要があります。データセットは次のようになっていることを思い出してください:

In [ ]:
titles_df.head()

`adapt` メソッドを使用して、データセット内のタイトルに対応する Pandas Series を直接使用してデータに適応させることができます:

In [ ]:
preprocessing_layer.adapt(titles_df.title)

これで、前処理レイヤに入力テキストを適用するだけで、テキストの整数表現を作成できるようになります:

In [ ]:
X_train = keras.ops.convert_to_tensor(titles_train)
X_valid = keras.ops.convert_to_tensor(titles_valid)
X_train[:5]

In [ ]:
integers = preprocessing_layer(X_train[:5])
integers

モデルに埋め込みレイヤを作成する際に `VOCAB_SIZE` が必要になります。この関数を使用して取得できます:

In [ ]:
VOCAB_SIZE = len(preprocessing_layer.get_vocabulary())
VOCAB_SIZE

## DNN モデルの構築

以下の build_dnn_model 関数は、単語の整数を密なベクトルに変換するシンプルな埋め込みレイヤと、各クラスの確率を返す Dense softmax レイヤを実装した、コンパイル済みの Keras モデルを返します。


埋め込みレイヤによって返された単語ベクトルの平均を計算するために、Embedding レイヤと Dense softmax レイヤの間にカスタムの Keras Lambda レイヤを配置する必要があることに注意してください。この平均が Dense softmax レイヤに入力されます。これを行うことで、シンプルでありながら単語の順序に関する情報が失われ、文を「Bag-of-Words（単語の袋）」として捉えるモデルが作成されます。

### ラボ課題

3 つのレイヤを持つ Keras Sequential モデルを作成します:
* 第 1 レイヤは入力レイヤ（Input レイヤ）にする必要があります
* 第 2 レイヤは上記で作成した前処理レイヤにします
* 第 3 レイヤは出力次元が `embed_dim` に等しい Embedding レイヤにする必要があります
* 第 4 レイヤは Lambda レイヤを使用して、`keras.ops.mean()` を使用して平均を計算し、文の Bag-of-Words 表現を作成する必要があります
* 最後のレイヤは Dense レイヤを使用して、サンプルがどのクラスに属するかを予測する必要があります

In [ ]:
def build_dnn_model(embed_dim):
    model = Sequential(
        [
            # TODO: Your code goes here
        ]
    )

    model.compile(
        optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
    )
    return model

以下では、モデルを 100 エポックでトレーニングしますが、`PATIENCE` で指定されたステップ数の間検証損失が改善しなかった場合にトレーニングを即座に停止する `EarlyStopping` コールバックを追加します。また、後で TensorBoard を使用してすべてのモデルを比較できるように、`model.fit` メソッドに TensorBoard コールバックも指定していることに注意してください。

In [ ]:
%%time

MODEL_DIR = os.path.join(LOGDIR, "dnn")
shutil.rmtree(MODEL_DIR, ignore_errors=True)

BATCH_SIZE = 300
EPOCHS = 100
EMBED_DIM = 10
PATIENCE = 5

dnn_model = build_dnn_model(embed_dim=EMBED_DIM)


dnn_history = dnn_model.fit(
    X_train,
    Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_valid, Y_valid),
    callbacks=[EarlyStopping(patience=PATIENCE), TensorBoard(MODEL_DIR)],
)

pd.DataFrame(dnn_history.history)[["loss", "val_loss"]].plot()
pd.DataFrame(dnn_history.history)[["accuracy", "val_accuracy"]].plot()

dnn_model.summary()

Keras 前処理レイヤを使用しているため、入力に対してモデルから直接予測を取得できます。

In [ ]:
"""
モデルから予測を取得する関数
model=トレーニング済みモデル名
instances=予測を取得するインスタンスのリスト
"""


def get_predictions(model, instances):
    prediction = model.predict(instances)
    title_prediction = list(zip(instances, prediction))
    for line, predict in title_prediction:
        print(
            "Title: {} \nGithub:{:.2%} \nNYTimes:{:.2%} \nTechCrunch:{:.2%}".format(
                line, float(predict[0]), float(predict[1]), float(predict[2])
            )
        )

In [ ]:
get_predictions(dnn_model, X_train[:5])

## RNN モデルの構築

以下の `build_rnn_model` 関数は、単一の `GRU` レイヤを備えたシンプルな RNN モデルを実装したコンパイル済みの Keras モデルを返します。これにより、文中の単語の順序が考慮されるようになります。

最初と最後のレイヤはシンプルな DNN モデルと同じです。

パディングされた単語（0 で表されます）がこのレイヤおよび後続のレイヤで無視されるように、`Embedding` レイヤで `mask_zero=True` を設定していることに注意してください。

### ラボ課題
記事のクラスを予測する RNN モデルを構築するために、以下のコードを完成させてください。以下のコードは上記で作成した DNN と似ていますが、ここでは文の Bag-of-Words 表現を使用する必要はありません。代わりに、埋め込みレイヤを RNN/LSTM/GRU レイヤに直接渡すことができます。

In [ ]:
def build_rnn_model(embed_dim, units):
    model = Sequential(
        [
            # TODO: Your code goes here
        ]
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0001),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

上記と同様に、Early Stopping を使用してモデルをトレーニングしましょう。

DNN モデルと同等の精度が得られますが、より少ないエポック数（約 20 エポックに対して約 3 エポック）で達成できることを確認してください:

In [ ]:
%%time

MODEL_DIR = os.path.join(LOGDIR, "rnn")
shutil.rmtree(MODEL_DIR, ignore_errors=True)

EPOCHS = 100
BATCH_SIZE = 300
EMBED_DIM = 10
UNITS = 16
PATIENCE = 2

rnn_model = build_rnn_model(embed_dim=EMBED_DIM, units=UNITS)

history = rnn_model.fit(
    X_train,
    Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_valid, Y_valid),
    callbacks=[EarlyStopping(patience=PATIENCE), TensorBoard(MODEL_DIR)],
)

pd.DataFrame(history.history)[["loss", "val_loss"]].plot()
pd.DataFrame(history.history)[["accuracy", "val_accuracy"]].plot()

rnn_model.summary()

In [ ]:
get_predictions(rnn_model, X_train[:5])

## CNN モデルの構築

以下の `build_cnn_model` 関数は、単一の `Conv1D` レイヤを備えたシンプルな CNN モデルを実装したコンパイル済みの Keras モデルを返します。これにより、文中の単語の順序が考慮されるようになります。

最初と最後のレイヤはシンプルな DNN モデルと同じですが、畳み込みレイヤと softmax レイヤの間に `Flatten` レイヤを追加する必要があります。

パディングされた単語（0 で表されます）がこのレイヤおよび後続のレイヤで無視されるように、`Embedding` レイヤで `mask_zero=True` を設定していることに注意してください。

### ラボ課題
テキスト分類用の CNN モデルを作成するために、以下のコードを完成させてください。このモデルは、埋め込みレイヤから開始するという点で以前のモデルと似ています。ただし、次のレイヤでは 1 次元畳み込みを通過し、最終的に全結合 Dense レイヤに渡す必要があります。`build_cnn_model` 関数の引数を使用して 1D 畳み込みレイヤを設定してください。

In [ ]:
def build_cnn_model(embed_dim, filters, ksize, strides):
    model = Sequential(
        [
            # TODO: Your code goes here
        ]
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0001),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

モデルをトレーニングしましょう。

ここでも、DNN モデルと同等の精度がはるかに少ないステップ数で得られることが確認できます。

In [ ]:
%%time

MODEL_DIR = os.path.join(LOGDIR, "cnn")
shutil.rmtree(MODEL_DIR, ignore_errors=True)

EPOCHS = 100
BATCH_SIZE = 300
EMBED_DIM = 5
FILTERS = 200
STRIDES = 2
KSIZE = 3
PATIENCE = 2


cnn_model = build_cnn_model(
    embed_dim=EMBED_DIM,
    filters=FILTERS,
    strides=STRIDES,
    ksize=KSIZE,
)

cnn_history = cnn_model.fit(
    X_train,
    Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_valid, Y_valid),
    callbacks=[EarlyStopping(patience=PATIENCE), TensorBoard(MODEL_DIR)],
)

pd.DataFrame(cnn_history.history)[["loss", "val_loss"]].plot()
pd.DataFrame(cnn_history.history)[["accuracy", "val_accuracy"]].plot()

cnn_model.summary()

In [ ]:
get_predictions(cnn_model, X_train[:5])

Copyright 2022 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License